In [1]:
def find_frequency(data):
    dec= {}
    for item in data:
        if item not in dec:
            dec[item] = 1
        else :
            dec[item] +=1
    return dec

In [3]:
print(find_frequency([101, 102, 101, 103, 102, 101]))

{101: 3, 102: 2, 103: 1}


In [22]:
def find_rolling_mean(arr,k):
    result = []
    length = len(arr)
    for indx, val in enumerate(arr):
        if indx+k <= length:
            result.append(round(sum(arr[indx : indx+k])/k,2))
        else:
            break
    return result

In [23]:
print(find_rolling_mean([100, 200, 150, 300, 250],3))

[150.0, 216.67, 233.33]


In [41]:
import string
import re

filter =  str.maketrans('','',string.punctuation)  

def preprocessing(data):
    result = ""
    new_data = data.split(" ")
    for word in new_data:
        new_word  =  word.translate(filter)
        result +=" "
        result += new_word.lower()
    return result

In [42]:
print(preprocessing("Hello World!!! ...123"))

 hello world 123


In [43]:
def find_product_unique(store1, store2):
    interesection = set()
    unique_store_1 = set()
    union = set()
    for item in store1:
        if item in store2:
            interesection.add(item)
        elif item not in store2:
            unique_store_1.add(item)
        union.add(item)
    for item in store2:
        union.add(item)

    return interesection,unique_store_1,union

In [44]:
print(find_product_unique(['electronics', 'clothing', 'food'],['electronics', 'toys', 'food']))

({'electronics', 'food'}, {'clothing'}, {'electronics', 'clothing', 'toys', 'food'})


In [1]:
raw_data = """
2024-01-15 09:23:45|user_a3x|post_12|like
2024-01-15 09:24:12|user_b7y|post_12|comment|Great content!
2024-01-15 09:25:33|user_a3x|post_45|share
2024-01-15 10:15:22|user_c9z|post_12|like
2024-01-15 10:16:45|user_a3x|post_12|comment|Thanks everyone
2024-01-15 11:22:10|user_d2w|post_45|like
2024-01-15 11:23:55|user_b7y|post_45|share
2024-01-15 12:30:15|user_c9z|post_78|like
2024-01-15 13:45:30|user_a3x|post_78|comment|Interesting
2024-01-15 14:20:18|user_e5v|post_12|like
""" 

** Each like = 1 point
Each comment = 3 points
Each share = 5 points
Bonus: If a user engages with the same post multiple times, multiply their subsequent engagements by 0.5 **

In [23]:
def find_top_post(data):
    data_lines = data.split("\n")
    new_data_lines = []
    for line in data_lines:
        if line != "":
            new_data_lines.append(line)
    post_dec = {}
    user_dec = {}
    for line in new_data_lines:
        data = line.split("|")
        if data[2] not in post_dec : 
            post_dec[data[2]] = 0
        if data[3] == "like" : 
            post_dec[data[2]]+=1 
        elif data[3] == "comment" :
            post_dec[data[2]] +=3
        elif data[3] == "share" :
            post_dec[data[2]] +=5
        if data[1] not in user_dec:
            user_dec[data[1]] =[data[2]] 
        else :
            if data[2] in user_dec[data[1]]:
                post_dec[data[2]]+=0.5
            else :
                user_dec[data[1]].append(data[2])
    post_dec = sorted(post_dec.items(), key=lambda item : item[1], reverse= True)
    return post_dec,user_dec

In [24]:
print(find_top_post(raw_data))

([('post_45', 11), ('post_12', 9.5), ('post_78', 4)], {'user_a3x': ['post_12', 'post_45', 'post_78'], 'user_b7y': ['post_12', 'post_45'], 'user_c9z': ['post_12', 'post_78'], 'user_d2w': ['post_45'], 'user_e5v': ['post_12']})


In [25]:
cart_data = """
order_001:[item_A:45.99:2, item_B:12.50:1, item_C:89.00:1]
order_002:[item_A:45.99:1, item_D:34.99:3]
order_003:[item_B:12.50:2, item_C:89.00:1, item_E:15.75:4]
order_004:[item_A:45.99:1, item_B:12.50:1]
order_005:[item_D:34.99:1, item_E:15.75:2, item_F:120.00:1]
order_006:[item_A:45.99:3, item_C:89.00:2]
order_007:[item_B:12.50:1, item_D:34.99:1, item_E:15.75:1]
"""

discount_rules = """
rule_1:buy_2_item_A_get_10%_off
rule_2:spend_150_get_15_off
rule_3:buy_item_C_and_item_A_together_get_20%_off_total
"""

**Parse the cart data and calculate total revenue per order
Identify which discount rule(s) apply to each order
Calculate the optimized discount (apply the rule that gives maximum savings to customer)
Return total revenue before and after discounts, and which rule was most profitable for the business (least discount given)**

In [104]:
def find_max_discount_revenue(cart_data, discount_rules):
    item_dec = {}
    customer_dec= {}
    line_split_rules = cart_data.splitlines()
    cart_value = []
    for line in line_split_rules:
        if line not in ["", None]:
            cart_value.append(line)
    for line in cart_value:
        s = line[10:]
        s = s.strip("[]").split(",")
        total_cost = 0
        all_item = {}
        for item in s :
            value = item.split(":")
            item_name = value[0].strip()
            amount = float(value[1])
            qty = int(value[2])
            all_item[item_name] = qty
            total_cost += (amount*qty)
            item_cost = round((amount*qty),2)
            if item_name not in item_dec:
                item_dec[item_name] = item_cost
            else :
                item_dec[item_name]+=item_cost
        def find_discount(dec,total_cost):
            reduce = 0
            for key,value in dec.items():
                if key == "item_A" and value >= 2 :
                    reduce+=10
            found = 2
            combined_offer = ["item_A", "item_c"]
            for key,value in dec.items():
                if key in combined_offer:
                    found -=1
            if found == 0 :
                reduce +=20 
            if total_cost >= 150 :
                reduce +=15
            return reduce
        # print(all_item)
        discount = find_discount(all_item, total_cost)
        final_cost = (total_cost - (total_cost /100) * discount)
        customer_name = line[:9]
        if customer_name not in customer_dec :
            customer_dec[customer_name] = round(final_cost,2)
            # print(customer_dec)
    return item_dec,customer_dec

In [103]:
print(find_max_discount_revenue(cart_data,discount_rules))

({'item_A': 321.93, 'item_B': 62.5, 'item_C': 356.0, 'item_D': 174.95000000000002, 'item_E': 110.25, 'item_F': 120.0}, {'order_001': 145.11, 'order_002': 128.32, 'order_003': 150.45, 'order_004': 58.49, 'order_005': 158.52, 'order_006': 236.98, 'order_007': 63.24})


In [74]:
def find_discount(dec,total_cost):
    reduce = 0
    for key,value in dec.items():
        if key == "item_A" and value >= 2 :
            reduce+=10
    found = 2
    combined_offer = ["item_A", "item_c"]
    for key,value in dec.items():
        if key in combined_offer:
            found -=1
    if found == 0 :
        reduce +=20 
    if total_cost >= 150 :
        reduce +=15
    return reduce

In [75]:
dec = {"item_A" : 2, "item_c" : 2}
reduce = 0 
found = 2
combined_offer = ["item_A", "item_c"]
for key,value in dec.items():
    if key in combined_offer:
        found -=1
if found == 0 :
    reduce +=20 
print(reduce)

20


In [76]:
print(find_discount(dec, 150))

45


In [108]:
s = " {Hello}"
print(s.strip(" "))

{Hello}


In [4]:
data = {
"order_001":["item_A:45.99:2", "item_B:12.50:1", "item_C:89.00:1"],
"order_002":["item_A:45.99:1", "item_D:34.99:3"],
"order_003":["item_B:12.50:2", "item_C:89.00:1", "item_E:15.75:4"],
"order_004":["item_A:45.99:1", "item_B:12.50:1"],
"order_005":["item_D:34.99:1", "item_E:15.75:2", "item_F:120.00:1"],
"order_006":["item_A:45.99:3", "item_C:89.00:2"], 
"order_007":["item_B:12.50:1", "item_D:34.99:1", "item_E:15.75:1"]
}

import pandas as pd 

df = pd.DataFrame(
    [(k, v) for k, v in data.items()],
    columns=["order_id", "order_items"]
)
df.head()

,order_id,order_items
0,order_001,"[item_A:45.99:2, item_B:12.50:1, item_C:89.00:1]"
1,order_002,"[item_A:45.99:1, item_D:34.99:3]"
2,order_003,"[item_B:12.50:2, item_C:89.00:1, item_E:15.75:4]"
3,order_004,"[item_A:45.99:1, item_B:12.50:1]"
4,order_005,"[item_D:34.99:1, item_E:15.75:2, item_F:120.00:1]"


In [3]:
d = [[1,"hi"],[2,"Hello"]]
f = pd.DataFrame(d, columns=["No", "word"])
f.head()

,No,word
0,1,hi
1,2,Hello
